In [1]:
import pandas as pd
import ipyparallel as ipp
import multiprocessing as mp
import time
import spacy 
import json
import os
from tqdm import tqdm
tqdm.pandas()


In [2]:
# Function to run the pipeline and return the result and time taken
def check_time(func):
    def sec_to_min(seconds):
        minutes = int(seconds // 60)
        remaining_seconds = round(seconds % 60)
        return f"{minutes:02}:{remaining_seconds:02}"
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time = sec_to_min(total_time)
        print("Results: ", result)
        print(f"Time taken: {total_time}")
        return result, total_time
    return wrapper

In [3]:
def run_basic_pipeline(text, has_location):
    import json

    unwanted_entities_path = "./geodata/unwanted_locations.json"  
    with open(unwanted_entities_path, 'r') as file:
        unwanted_entities = json.load(file)

    if (has_location):
        return None
    
    if (text == None or text == ""):
        return None
    
    try: 
        import spacy

        # Load the spacy model with the span_marker pipeline component
        nlp = spacy.load("en_core_web_sm", exclude=["ner"])
        nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})
                        
        # Return a valid location if any
        entities = nlp(text).ents
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
                        
    except Exception as error:
        print(error)
        return error

In [4]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

unwanted_entities_path = "./geodata/unwanted_locations.json"  
with open(unwanted_entities_path, 'r') as file:
    unwanted_entities = json.load(file)

@check_time
def run_series_basic_pipeline(article):
    
    if (article['Explicit_Pass'] is not None):
        return None
    
    text = article['body']
    
    if (text == None or text == ""):
        return None
    
    try:                
        # Return a valid location if any
        entities = nlp(text).ents
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
                        
    except Exception as error:
        print(error)
        return error

In [5]:
@check_time
def handle_basic_pipeline(articles):
    return articles.progress_apply(run_series_basic_pipeline, axis=1)

In [6]:
def run_chunk_pipeline(text, has_location):
    chunk_size = 100
    import json

    unwanted_entities_path = "./geodata/unwanted_locations.json"  
    with open(unwanted_entities_path, 'r') as file:
        unwanted_entities = json.load(file)

    if (has_location):
        return None
    
    if (text == None or text == ""):
        return None
    
    try: 
        import spacy

        # Load the spacy model with the span_marker pipeline component
        nlp = spacy.load("en_core_web_sm", exclude=["ner"])
        nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

        words = text.split()
        chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

        # Process each chunk and return if a valid facilty is found
        for chunk in chunks:
            entities = nlp(chunk).ents
            for entity in entities:
                # If it's a valid facility, return it
                if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                    return entity.text
    
        return None
                        
    except Exception as error:
        print(error)
        return error

In [7]:
chunk_size = 100

@check_time
def run_chunking_pipeline(article):
    
    if (article['Explicit_Pass'] is not None):
        return None
    
    text = article['body']
    
    if (text == None or text == ""):
        return None
    
    try:                
        words = text.split()
        chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

        # Process each chunk and return if a valid facilty is found
        for chunk in chunks:
            entities = nlp(chunk).ents
            for entity in entities:
                # If it's a valid facility, return it
                if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                    return entity.text
    
        return None
                        
    except Exception as error:
        print(error)
        return error


In [8]:
@check_time
def handle_chunk_pipeline(articles):
    return articles.progress_apply(run_chunking_pipeline, axis=1)

In [9]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

In [10]:
@check_time
def run_multiprocessing(data, function, cpu_count):
    # Step 1: Clean up any existing IPyParallel processes
    try:
        os.system('ipcluster stop --profile=default')
    except Exception as e:
        print(f"Error stopping existing cluster: {e}")

    # Start and connect to an IPyParallel cluster
    rc = ipp.Cluster(n=cpu_count, controller_args=['--debug']).start_and_connect_sync()
    dview = rc[:]

    has_location_list = [x is not None for x in data['Explicit_Pass']]
    text_list = data['body'].tolist()
    try: 
        # Process the articles parallelly using ipyparallel
        valid_entity_list = dview.map_sync(function, text_list, has_location_list)
    except Exception as e:
        print(f"Error processing articles: {e}")
        rc.close()
        return None
    rc.close()
    return valid_entity_list
        

In [11]:
def run_tests(data):
    # Run multiprocessing pipeline with different number of workers and compare results
    results_df = data.copy()
    time_dict = {}

    # Test multiprocessing alone
    # for cpu_count in [5]: #  range(2, mp.cpu_count() + 1, 2):
    #     print(f"Running multiprocessing pipeline with {cpu_count} workers...")
    #     results, total_time = run_multiprocessing(data, run_basic_pipeline, cpu_count)
    #     results_df[f"Multi_CPU_{cpu_count}"] = results
    #     time_dict[f"Multi_CPU_{cpu_count}"] = total_time
    
    # Test chunk processing alone
    print("Running chunk pipeline...")
    results, total_time = handle_chunk_pipeline(data)
    results_df['Chunk'] = results
    time_dict['Chunk'] = total_time  

    # Test basic serial processing run
    print("Running basic pipeline...")
    results, total_time = handle_basic_pipeline(data)
    results_df['Basic'] = results
    time_dict['Basic'] = total_time  

    # Test multiprocessing with chunks
    for cpu_count in [5]: #  range(2, mp.cpu_count() + 1, 2):
        print(f"Running multiprocessing chunk pipeline with {cpu_count} workers...")
        results, total_time = run_multiprocessing(data, run_chunk_pipeline, cpu_count)
        results_df[f"MultiChunk_CPU_{cpu_count}"] = results
        time_dict[f"MultiChunk_CPU_{cpu_count}"] = total_time

    return results_df, time_dict

In [12]:
article_df = pd.read_csv("sample_data/cleaned_sample_data.csv")
article_df["Explicit_Pass"] = [None, None, "Location", None] # Add a few explicit locations for testing
# article_df["Explicit_Pass"] = [None] # Add a few explicit locations for testing

# df = run_tests(article_df)

In [13]:
# df

In [14]:
# df.to_csv("sample_data/results_2.csv", index=False)

In [15]:
import re
from bs4 import BeautifulSoup

sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one

# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]     
 

In [16]:
def clean_up_sample(raw_df):
    df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

    df = df.drop_duplicates(subset=['hl1'])

    # Function to extract the text from the html of the article
    func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
    df['body'] = df['body'].progress_apply(func_clean_html)
    df['hl1'] = df['hl1'].progress_apply(func_clean_html)

    # Function to remove extra symbols from the text
    func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
    df['body'] = df['body'].progress_apply(func_clean_regex)
    df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

    return df 

def pick_sample(articles, sample_size):
    # Pick a random sample of articles
    raw_df = articles.sample(sample_size)

    sample_df = clean_up_sample(raw_df)

    return sample_df


In [17]:
def run_test_batches(test_amount, sample_size, articles_df):
    time_df = pd.DataFrame(columns=['Basic', 'Multi_CPU_2', 'Chunk', 'MultiChunk_CPU_5'])

    # Run the tests multiple times 
    for test_count in range(test_amount):
        print(f"Running test {test_count + 1}...")

        # Pick a random sample of articles
        sample_df = pick_sample(articles_df, sample_size)
        sample_df["Explicit_Pass"] = None

        results_df, time_dict = run_tests(sample_df)
        time_row = pd.DataFrame([time_dict])
        time_df = pd.concat([time_df, time_row], ignore_index=True)

        print(results_df.to_string())
        print(time_df.to_string())
        results_df.to_csv(f"sample_data/results_{sample_size}_samples_run_{test_count + 1}.csv", index=False)
    time_df.to_csv(f"sample_data/time_{sample_size}_samples.csv", index=False)

In [18]:
run_test_batches(1, 100, full_df)

Running test 1...


  0%|          | 0/100 [00:00<?, ?it/s]C:\Users\axel0\AppData\Local\Temp\ipykernel_16864\1136650839.py:7: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
100%|██████████| 100/100 [00:00<?, ?it/s]


Running chunk pipeline...


  2%|▏         | 2/100 [01:20<1:05:33, 40.14s/it]

Results:  None
Time taken: 01:20


...

100%|██████████| 100/100 [2:17:05<00:00, 82.25s/it]

Results:  None
Time taken: 01:06
Results:  545                  (None, 01:13)
2234                 (None, 00:10)
8759                 (None, 03:02)
9306                 (None, 02:23)
3866     (Boston City Hall, 01:07)
                   ...            
2760                 (None, 00:03)
10716                (None, 01:11)
7000                 (None, 00:48)
7710                 (None, 00:41)
1007                 (None, 01:06)
Length: 100, dtype: object
Time taken: 137:05
Running multiprocessing chunk pipeline with 5 workers...


Starting 5 engines with <class 'ipyparallel.cluster.launcher.LocalEngineSetLauncher'>


  0%|          | 0/5 [00:00<?, ?engine/s]

Output for 3:
2024-07-11 03:02:17.412 [IPEngine] Loading connection info from $IPP_CONNECTION_INFO
2024-07-11 03:02:17.412 [IPEngine] WARNING | Not using CurveZMQ security
2024-07-11 03:02:17.432 [IPEngine] Registering with controller at tcp://127.0.0.1:56863
2024-07-11 03:02:17.434 [IPEngine] Shell_addrs: ['tcp://127.0.0.1:56865', 'tcp://127.0.0.1:56866', 'tcp://127.0.0.1:56871']
2024-07-11 03:02:17.434 [IPEngine] Connecting shell to tcp://127.0.0.1:56865
2024-07-11 03:02:17.434 [IPEngine] Connecting shell to tcp://127.0.0.1:56866
2024-07-11 03:02:17.434 [IPEngine] Connecting shell to tcp://127.0.0.1:56871
2024-07-11 03:02:17.434 [IPEngine] Starting nanny
Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 112, in _get_module_details
  File "C:\Users\axel0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\ipyparallel\__init__.py", line 11, in

In [ ]:
data = {
    'body': [
        "The new medical center, Boston General Hospital, has opened its doors to the public this week, offering state-of-the-art medical services to the community.",
        "The conference at Stanford University was a success, bringing together experts from various fields to discuss advancements in artificial intelligence.",
        "Central Park Zoo announced the birth of a rare white tiger, attracting visitors from all over the world to see the new addition.",
        "Microsoft's headquarters in Redmond are known for their innovation and cutting-edge technology development.",
        "The annual tech summit held at Silicon Valley was attended by representatives from Google, Apple, and Facebook.",
        "The renovation of the Los Angeles Public Library has been completed, providing improved facilities for reading and research.",
        "The seminar on climate change at Harvard University was well-received, with prominent scientists presenting their latest research findings.",
        "The opening of the new wing at the Smithsonian Museum has drawn large crowds eager to see the latest exhibits.",
        "Mayo Clinic in Rochester is renowned for its advanced medical treatments and patient care.",
        "The art exhibition at the Louvre Museum in Paris features works from renowned artists across different centuries."
    ]
}

